# MNIST rectified flow — analyze any wandb run

Set `RUN_ID` to any run trained by `train.py`. Everything (the frozen autoencoder, the
`VelocityDiT` velocity field, and the latent mean/std) is rebuilt from the run's
checkpoint alone — its saved hyperparameters carry the model + AE configs, and the AE
weights live in the checkpoint as a submodule — so no wandb round-trip is needed (the
checkpoint is located locally first, else downloaded as the run's model artifact).

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import matplotlib.pyplot as plt

from chimera.models import ConvAutoEncoder
from chimera.utils.experiment import find_ckpt
from train import NUM_CLASSES, OUTPUTS, PROJECT_DEFAULT, LitRectifiedFlow

RUN_ID = "PASTE_RUN_ID_HERE"
PROJECT = PROJECT_DEFAULT
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Rebuild the whole run from its checkpoint alone: load_from_checkpoint restores the
# VelocityDiT, the frozen AE submodule weights, and the latent mean/std buffers from the
# saved hyperparameters + state_dict. We only re-supply the AE *skeleton* (it's excluded
# from the hparams) built from the checkpoint's own ae_config; its weights are then loaded.
ckpt_path = find_ckpt(RUN_ID, PROJECT, OUTPUTS)
ae_config = torch.load(ckpt_path, map_location="cpu", weights_only=False)[
    "hyper_parameters"
]["ae_config"]
module = (
    LitRectifiedFlow.load_from_checkpoint(
        ckpt_path, autoencoder=ConvAutoEncoder(**ae_config)
    )
    .to(device)
    .eval()
)

print("flow params:", sum(p.numel() for p in module.model.parameters()))
print("latent shape:", module.latent_shape)

In [ ]:
# Sampling goes through LitRectifiedFlow.sample -- the exact integrator + CFG that train.py
# logs at validation -- overriding steps/guidance_scale per call. Fixed seed (own generator)
# so grids are comparable across scales.
@torch.inference_mode()
def sample(y, *, steps=50, guidance_scale=3.0, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    z0 = torch.randn(y.shape[0], module.latent_dim, device=device, generator=g)
    return module.sample(y, z0, guidance_scale=guidance_scale, steps=steps).cpu()


def show(imgs, rows, cols, row_labels=None):
    fig, axes = plt.subplots(rows, cols, figsize=(cols, rows), squeeze=False)
    for i in range(rows):
        for j in range(cols):
            ax = axes[i, j]
            ax.imshow(imgs[i * cols + j, 0], cmap="gray")
            ax.set_xticks([])
            ax.set_yticks([])
        if row_labels is not None:
            axes[i, 0].set_ylabel(row_labels[i], rotation=0, ha="right", va="center")
    plt.tight_layout()
    plt.show()

In [ ]:
# Class-conditioned samples: one row per digit, N samples each.
N = 8
y = torch.arange(NUM_CLASSES, device=device).repeat_interleave(N)
imgs = sample(y, steps=50, guidance_scale=3.0)
show(imgs, NUM_CLASSES, N, row_labels=[str(d) for d in range(NUM_CLASSES)])

In [ ]:
# Guidance-scale sweep: same fixed noise, one row per CFG scale, digits 0-9 across columns.
# Higher scale -> sharper, more class-faithful digits (less diverse).
scales = [1.0, 2.0, 3.0, 5.0]
y = torch.arange(NUM_CLASSES, device=device)
rows = [sample(y, steps=50, guidance_scale=s, seed=0) for s in scales]
imgs = torch.cat(rows, dim=0)
show(imgs, len(scales), NUM_CLASSES, row_labels=[f"cfg={s}" for s in scales])